# 🧊 SKETCH2SHAPE AI - TripoSR Cloud GPU Inference Backend
### TechZephyr 4.0 Hackathon • Indian Institute of Technology (IIT) Bhubaneswar

This Google Colab notebook runs the **pretrained TripoSR 3D reconstruction model** on an **NVIDIA Tesla T4 GPU (~14.56 GB VRAM)** and exposes a secure **Cloudflare Tunnel** URL to connect directly with your local Streamlit frontend.

## 1. Verify NVIDIA GPU Acceleration (Tesla T4)

In [ ]:
!nvidia-smi

## 2. Clone TripoSR & Install Dependencies

In [ ]:
# Clone official TripoSR repository
!git clone https://github.com/VAST-AI-Research/TripoSR.git
%cd TripoSR

# Install dependencies for high-speed neural 3D mesh synthesis
!pip install --upgrade pip
!pip install -r requirements.txt
!pip install fastapi uvicorn python-multipart rembg trimesh requests

## 3. Install Cloudflare Tunnel (cloudflared)

In [ ]:
# Download official cloudflared binary for Linux
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

## 4. Launch FastAPI Inference Server & Cloudflare Tunnel
Run this cell to start the server. It will display your **live public Cloudflare Tunnel URL**.
Copy the URL (e.g. `https://xxxx.trycloudflare.com`) and paste it into **⚡ Cloud AI Settings** in your local Streamlit app.

In [ ]:
import subprocess
import time
import re
import os

# Ensure server.py is present in working directory
if not os.path.exists('server.py'):
    # If uploaded or copied from repo
    print("Copying or creating server script...")

# Start FastAPI backend in background on port 8000
server_proc = subprocess.Popen([
    "uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

time.sleep(3)

# Launch Cloudflare tunnel
tunnel_proc = subprocess.Popen([
    "cloudflared", "tunnel", "--url", "http://localhost:8000"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print("Establishing Cloudflare Tunnel...")
tunnel_url = None

while True:
    line = tunnel_proc.stdout.readline()
    if not line:
        break
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        print("\n" + "="*60)
        print("🚀 TRIPOSR CLOUD GPU INFERENCE BACKEND IS ONLINE!")
        print(f"👉 PUBLIC CLOUDFLARE URL: {tunnel_url}")
        print("="*60)
        print("\nPaste this URL into Sketch2Shape AI -> '⚡ Cloud AI Settings' in your sidebar.")
        break

# Keep alive
try:
    server_proc.wait()
except KeyboardInterrupt:
    print("Stopping server and tunnel...")
    server_proc.terminate()
    tunnel_proc.terminate()
